In [9]:
!pip install -q pandas numpy scikit-learn nltk ipywidgets

import pandas as pd
import numpy as np
import re
import nltk

from google.colab import files
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, clear_output

nltk.download('stopwords')

from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

print("Libraries loaded successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 25.0 MB/s eta 0:00:00
Libraries loaded successfully!


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [10]:
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Saving bbc_news.csv to bbc_news (1).csv
Dataset loaded successfully!
Shape: (42115, 5)

Columns:
['title', 'pubDate', 'guid', 'link', 'description']


,title,pubDate,guid,link,description
0,Ukraine: Angry Zelensky vows to punish Russian...,"Mon, 07 Mar 2022 08:01:56 GMT",https://www.bbc.co.uk/news/world-europe-60638042,https://www.bbc.co.uk/news/world-europe-606380...,The Ukrainian president says the country will ...
1,War in Ukraine: Taking cover in a town under a...,"Sun, 06 Mar 2022 22:49:58 GMT",https://www.bbc.co.uk/news/world-europe-60641873,https://www.bbc.co.uk/news/world-europe-606418...,"Jeremy Bowen was on the frontline in Irpin, as..."
2,Ukraine war 'catastrophic for global food',"Mon, 07 Mar 2022 00:14:42 GMT",https://www.bbc.co.uk/news/business-60623941,https://www.bbc.co.uk/news/business-60623941?a...,One of the world's biggest fertiliser firms sa...
3,Manchester Arena bombing: Saffie Roussos's par...,"Mon, 07 Mar 2022 00:05:40 GMT",https://www.bbc.co.uk/news/uk-60579079,https://www.bbc.co.uk/news/uk-60579079?at_medi...,The parents of the Manchester Arena bombing's ...
4,Ukraine conflict: Oil price soars to highest l...,"Mon, 07 Mar 2022 08:15:53 GMT",https://www.bbc.co.uk/news/business-60642786,https://www.bbc.co.uk/news/business-60642786?a...,Consumers are feeling the impact of higher ene...


In [11]:
# Convert column names to lowercase
df.columns = df.columns.str.strip().str.lower()

# Fill missing values
df['title'] = df['title'].fillna('')
df['description'] = df['description'].fillna('')

# Create document using title + description
df['document'] = (
    df['title'].astype(str) + ' ' +
    df['description'].astype(str)
)

# Remove empty documents
df = df[
    df['document'].str.strip() != ''
].reset_index(drop=True)

print("Number of text documents:", len(df))

display(
    df[['title', 'description', 'document']].head()
)

Number of text documents: 42115


,title,description,document
0,Ukraine: Angry Zelensky vows to punish Russian...,The Ukrainian president says the country will ...,Ukraine: Angry Zelensky vows to punish Russian...
1,War in Ukraine: Taking cover in a town under a...,"Jeremy Bowen was on the frontline in Irpin, as...",War in Ukraine: Taking cover in a town under a...
2,Ukraine war 'catastrophic for global food',One of the world's biggest fertiliser firms sa...,Ukraine war 'catastrophic for global food' One...
3,Manchester Arena bombing: Saffie Roussos's par...,The parents of the Manchester Arena bombing's ...,Manchester Arena bombing: Saffie Roussos's par...
4,Ukraine conflict: Oil price soars to highest l...,Consumers are feeling the impact of higher ene...,Ukraine conflict: Oil price soars to highest l...


In [12]:
def preprocess(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # Remove special characters and numbers
    text = re.sub(r'[^a-z\s]', ' ', text)

    # Tokenization
    words = text.split()

    # Remove stopwords
    words = [
        word for word in words
        if word not in stop_words
    ]

    return ' '.join(words)


df['processed_document'] = df['document'].apply(preprocess)

display(
    df[['title', 'processed_document']].head()
)

,title,processed_document
0,Ukraine: Angry Zelensky vows to punish Russian...,ukraine angry zelensky vows punish russian atr...
1,War in Ukraine: Taking cover in a town under a...,war ukraine taking cover town attack jeremy bo...
2,Ukraine war 'catastrophic for global food',ukraine war catastrophic global food one world...
3,Manchester Arena bombing: Saffie Roussos's par...,manchester arena bombing saffie roussos parent...
4,Ukraine conflict: Oil price soars to highest l...,ukraine conflict oil price soars highest level...


In [13]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    min_df=1,
    max_df=0.95
)

tfidf_matrix = vectorizer.fit_transform(
    df['processed_document']
)

terms = vectorizer.get_feature_names_out()

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Number of documents:", tfidf_matrix.shape[0])
print("Number of terms:", tfidf_matrix.shape[1])

TF-IDF matrix shape: (42115, 10000)
Number of documents: 42115
Number of terms: 10000


In [14]:
tfidf_table = pd.DataFrame(
    tfidf_matrix[:10].toarray(),
    columns=terms
)

tfidf_table.insert(
    0,
    'Document',
    range(1, 11)
)

display(tfidf_table.iloc[:, :31])

,Document,aa,aaron,abandoned,abandons,abba,abbey,abbott,abdominal,abducted,...,abortions,abramovich,abroad,absence,absent,absolute,absolutely,abu,abualouf,abuse
0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [15]:
def search_documents(query, top_k=10):

    # Preprocess query
    processed_query = preprocess(query)

    # Convert query into TF-IDF vector
    query_vector = vectorizer.transform(
        [processed_query]
    )

    # Calculate cosine similarity
    similarity_scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    # Sort from highest similarity to lowest
    ranked_indices = np.argsort(
        similarity_scores
    )[::-1]

    # Select top documents
    ranked_indices = ranked_indices[:top_k]

    # Create result dataframe
    results = df.iloc[ranked_indices].copy()

    results['Similarity Score'] = (
        similarity_scores[ranked_indices]
    )

    results['Rank'] = range(
        1,
        len(results) + 1
    )

    # Select columns to display
    results = results[
        [
            'Rank',
            'title',
            'description',
            'Similarity Score'
        ]
    ]

    return results

In [16]:
query = "Ukraine Russia war"

results = search_documents(
    query,
    top_k=10
)

display(results)

,Rank,title,description,Similarity Score
14167,1,Has Putin's war failed and what does Russia wa...,"A year into Russia's war, he has little to sho...",0.615937
3924,2,Nadya Karpova: The Russia striker speaking out...,Nadya Karpova is one of very few Russian sport...,0.577666
27453,3,Ukraine war: What Russia's escalating air atta...,"In just five days, Russia has fired 500 missil...",0.553220
1171,4,Why Russia is trying to encircle Ukraine's east,Russia has shifted most of the focus of its wa...,0.552659
285,5,Ukraine war: Ukrainians mourn their fallen sol...,Funerals are held for the first Ukrainian vict...,0.542043
957,6,Why has Russia invaded Ukraine and what does P...,Weeks after Russia's leader unleashed war on U...,0.532846
1132,7,Why has Russia invaded Ukraine and what does P...,Weeks after Russia's leader unleashed war on U...,0.532846
1864,8,Why has Russia invaded Ukraine and what does P...,Weeks after Russia's leader unleashed war on U...,0.532846
1753,9,Why has Russia invaded Ukraine and what does P...,Weeks after Russia's leader unleashed war on U...,0.532846
2334,10,Why has Russia invaded Ukraine and what does P...,More than two months after Russia's leader unl...,0.527218


In [17]:
def show_query_tfidf(query):

    processed_query = preprocess(query)

    query_vector = vectorizer.transform(
        [processed_query]
    )

    query_values = query_vector.toarray()[0]

    vocabulary = vectorizer.vocabulary_

    query_terms = processed_query.split()

    valid_terms = [
        term for term in query_terms
        if term in vocabulary
    ]

    if len(valid_terms) == 0:
        print("No query terms found in the dataset.")
        return

    query_table = pd.DataFrame({
        'Term': valid_terms,
        'Query TF-IDF': [
            query_values[vocabulary[term]]
            for term in valid_terms
        ]
    })

    print("TF-IDF VALUES OF USER QUERY")
    print("=" * 50)

    display(query_table)

In [18]:
show_query_tfidf(
    "Ukraine Russia war"
)

TF-IDF VALUES OF USER QUERY


,Term,Query TF-IDF
0,ukraine,0.540899
1,russia,0.620836
2,war,0.567442


In [19]:
def show_document_tfidf(query, top_k=10):

    processed_query = preprocess(query)

    query_terms = processed_query.split()

    vocabulary = vectorizer.vocabulary_

    valid_terms = [
        term
        for term in query_terms
        if term in vocabulary
    ]

    if len(valid_terms) == 0:
        print("No query terms found in the dataset.")
        return

    term_indices = [
        vocabulary[term]
        for term in valid_terms
    ]

    # Get TF-IDF values
    values = tfidf_matrix[
        :,
        term_indices
    ].toarray()

    table = pd.DataFrame(
        values,
        columns=valid_terms
    )

    table.insert(
        0,
        'Document',
        df['title'].values
    )

    # Sum of TF-IDF values for query terms
    table['Total TF-IDF'] = table[
        valid_terms
    ].sum(axis=1)

    # Sort highest first
    table = table.sort_values(
        'Total TF-IDF',
        ascending=False
    )

    print("DOCUMENT TF-IDF TABLE")
    print("=" * 80)

    display(
        table.head(top_k)
    )

In [20]:
show_document_tfidf(
    "Ukraine Russia war",
    top_k=10
)

DOCUMENT TF-IDF TABLE


,Document,ukraine,russia,war,Total TF-IDF
14167,Has Putin's war failed and what does Russia wa...,0.195124,0.447920,0.409398,1.052441
3924,Nadya Karpova: The Russia striker speaking out...,0.387051,0.222126,0.406045,1.015222
27453,Ukraine war: What Russia's escalating air atta...,0.299236,0.343459,0.313920,0.956616
285,Ukraine war: Ukrainians mourn their fallen sol...,0.363183,0.208428,0.381005,0.952616
1171,Why Russia is trying to encircle Ukraine's east,0.356294,0.408950,0.186889,0.952133
1132,Why has Russia invaded Ukraine and what does P...,0.343521,0.394289,0.180189,0.918000
1753,Why has Russia invaded Ukraine and what does P...,0.343521,0.394289,0.180189,0.918000
1864,Why has Russia invaded Ukraine and what does P...,0.343521,0.394289,0.180189,0.918000
957,Why has Russia invaded Ukraine and what does P...,0.343521,0.394289,0.180189,0.918000
2334,Why has Russia invaded Ukraine and what does P...,0.339893,0.390124,0.178286,0.908304


In [21]:
import ipywidgets as widgets

query_box = widgets.Text(
    placeholder='Enter your search query...',
    description='Query:',
    layout=widgets.Layout(width='80%')
)

top_k_box = widgets.IntSlider(
    value=10,
    min=1,
    max=20,
    step=1,
    description='Top K:'
)

search_button = widgets.Button(
    description='SEARCH',
    button_style='primary'
)

output = widgets.Output()


def perform_search(button):

    with output:

        clear_output()

        query = query_box.value.strip()

        if query == '':
            print("Please enter a query.")
            return

        print("=" * 90)
        print("TF-IDF INFORMATION RETRIEVAL SYSTEM")
        print("=" * 90)

        print("\nUSER QUERY:")
        print(query)

        # Query TF-IDF
        print("\n1. QUERY TF-IDF")
        print("-" * 90)

        show_query_tfidf(query)

        # Ranked documents
        print("\n2. RANKED DOCUMENTS")
        print("-" * 90)

        results = search_documents(
            query,
            top_k=top_k_box.value
        )

        display(results)

        # Document TF-IDF
        print("\n3. DOCUMENT TF-IDF TABLE")
        print("-" * 90)

        show_document_tfidf(
            query,
            top_k=top_k_box.value
        )


search_button.on_click(
    perform_search
)

display(
    query_box,
    top_k_box,
    search_button,
    output
)

Text(value='', description='Query:', layout=Layout(width='80%'), placeholder='Enter your search query...')

IntSlider(value=10, description='Top K:', max=20, min=1)

Button(button_style='primary', description='SEARCH', style=ButtonStyle())

Output()